### 🤖 Guía de Asistencia con IA: IA como Copiloto de Programación

En esta notebook, utilizaremos la Inteligencia Artificial como copiloto de aprendizaje. Aquí tienes algunas sugerencias de cómo interactuar con un asistente de IA (como Gemini o ChatGPT):

- **Explicación de código**: Si encuentras una función o línea que no comprendes, pregúntale a la IA: *'Explica qué hace esta función línea por línea en Python: [pega tu código]'*.
- **Corrección de errores**: Si tienes un error al ejecutar, copia el error completo y pregúntale: *'¿Por qué este código genera este error y cómo lo soluciono? [pega tu código y error]'*.
- **Optimización**: Pregunta: *'¿Cómo puedo escribir este código de forma más eficiente y limpia?'*.



# Programa Ingenia+ Data Science

Recordemos que un proyecto de data science tiene varias etapas:

1. Recolección de Datos
2. Exploración y Procesamiento de los datos
3. Modelado
4. Puesta en Producción

En clases anteriores, trabajamos con el dataset `StudentPerformace`. Pudimos observar el tipo de datos que teniamos y le hiciemos algunas transformaciones.

Llego la hora de utilizar los conceptos de aprendizaje supervisado para hacer predicciones sobre alguna variable en el dataset.

Para poder ajustar modelos de Machine Learning, la herramienta más sencilla y más utilizada es `scikit-learn`. Esta librería es muy utilizada ya que contiene funciones desarrolladas para la mayoría de los modelos de machine learning. Además `Scikit-learn` tienen una amplia y muy buena [documentación](https://scikit-learn.org/stable/).

<font size=5> 🚀 👩🏽‍💻 Machine Learning: Aprendizaje Supervisado 📣</font>

In [57]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.cm as cm

In [58]:
import os

In [59]:
os.getcwd()

'/home/leaolarte/Curso_YPF_Data_Science'

Primero, vamos a leer los datos limpios y procesados.

In [60]:
# Leemos nuevamente los datos de los estudiantes
students = pd.read_csv("DataSets/StudentsPerformance.csv")

In [61]:
students.head()

,Unnamed: 0,gender,race/ethnicity,parental level of education,lunch,test preparation course,math score,reading score,writing score
0,0,female,group B,bachelor's degree,standard,none,7.2,72,74
1,1,female,group C,some college,standard,completed,6.9,90,88
2,2,female,group B,master's degree,standard,none,9.0,95,93
3,3,male,group A,associate's degree,free/reduced,none,4.7,57,44
4,4,male,group C,some college,standard,none,7.6,78,75


Antes que nada, chequeemos nuevamente que eliminamos todos los valores faltantes.

In [62]:
students.isnull().sum()

Unnamed: 0                     0
gender                         0
race/ethnicity                 0
parental level of education    0
lunch                          0
test preparation course        0
math score                     0
reading score                  0
writing score                  0
dtype: int64

Ahora tenemos que decidir que variable o variables vamos a predecir. Esto quiere decir que tenemos que elegir cual va a ser la variable que al predecirla conteste la pregunta que nos hicimos de los datos.

Lo que queremos saber es:

1. ¿Qué puntaje final obtendra un estudiante?
2. Dado un estudiante, ¿aprobara o no el examen?

**_¿Qué tipo de problemas estamos tratando de resolver?_**

Para contestar estas preguntas vamos a obtener dos nuevas variables:

1. El puntaje final: Lo obtendremos promediando los tres puntajes (matematica, lectura y escritura)
2. Aprobado: Una etiqueta que me permita saber si cada estudiante del dataset aprobo el examen (puntaje final >= 60) o no lo aprobo (puntaje final < 60).

In [63]:
students['puntaje_final'] = (students['math score'] + students['reading score'] + students['writing score']) / 3

In [64]:
students.head()

,Unnamed: 0,gender,race/ethnicity,parental level of education,lunch,test preparation course,math score,reading score,writing score,puntaje_final
0,0,female,group B,bachelor's degree,standard,none,7.2,72,74,51.066667
1,1,female,group C,some college,standard,completed,6.9,90,88,61.633333
2,2,female,group B,master's degree,standard,none,9.0,95,93,65.666667
3,3,male,group A,associate's degree,free/reduced,none,4.7,57,44,35.233333
4,4,male,group C,some college,standard,none,7.6,78,75,53.533333


## 📈 Regresión

Comenzaremos contestando la pregunta 1. Este problema es un problema de regresión (_¿Por qué?_)

porque utiliza variables numericas.

### Seleccion de target (y) y variables (X)

Lo primero que haremos, es seleccionar las variables o features que deseemos usar como variable independientes (aquellas que van a explicar mi predicción) y la variable a predecir.

Usaremos todas las variables menos aquellas que fueron usadas para calcular el puntaje final. Debemos también asegurarnos de no incluir las variables a predecir. Como variable a predecir usaremos el puntaje final.

In [65]:
# Elimino aquellas variables que no quiero incluir en el modelo y las guardo en x.
x = students.drop(['math score', 'reading score', 'writing score', 'puntaje_final'],
                  axis=1)

In [66]:
x.head()

,Unnamed: 0,gender,race/ethnicity,parental level of education,lunch,test preparation course
0,0,female,group B,bachelor's degree,standard,none
1,1,female,group C,some college,standard,completed
2,2,female,group B,master's degree,standard,none
3,3,male,group A,associate's degree,free/reduced,none
4,4,male,group C,some college,standard,none


In [67]:
lista_atributos = x.columns

In [68]:
# Ahora selecciono las etiquetas y las guardo en y.
y = students['puntaje_final']

In [69]:
#x son los nuevos atributos, y la variable a pr4edecir
x, y = np.array(x), np.array(y)

In [70]:
x

array([[0, 'female', 'group B', "bachelor's degree", 'standard', 'none'],
       [1, 'female', 'group C', 'some college', 'standard', 'completed'],
       [2, 'female', 'group B', "master's degree", 'standard', 'none'],
       ...,
       [997, 'female', 'group C', 'high school', 'free/reduced',
        'completed'],
       [998, 'female', 'group D', 'some college', 'standard',
        'completed'],
       [999, 'female', 'group D', 'some college', 'free/reduced', 'none']],
      shape=(1000, 6), dtype=object)

In [71]:
y[:10]

array([51.06666667, 61.63333333, 65.66666667, 35.23333333, 53.53333333,
       56.03333333, 65.26666667, 28.66666667, 45.8       , 37.93333333])

### Separando Train / Test

Ahora tenemos que separar el dataset en conjuntos de entrenamiento (X_train, y_train) y de testeo (X_test, y_test) usando la función train_test_split de scikit-learn.

¿Por qué separamos el dataset? Porque es necesario entrenar el modelo con un set de datos. El modelo no puede ver los datos de evaluación cuando se esta entrenando!

La forma en que se dividen los datos normalmente es 80% entrenamiento -20% evaluación)

In [72]:
# Importamos la librearia para separar el dataset.
from sklearn.model_selection import train_test_split

In [73]:
X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.2,
                                                    random_state=42)

In [74]:
x.shape

(1000, 6)

In [75]:
X_train.shape

(800, 6)

In [76]:
X_test.shape

(200, 6)

### Random Forest

Recordemos que Random Forest es un algoritmo de ensamble. Hay dos tecnicas para esto: Bagging y Boosting.

El objetivo de bagging es entrenar distintos modelos, donde cada uno vea distintas porciones del set de entrenamiento. Random Forest, además de aplicar Bagging, también selecciona features al azar, de esa manera descorrelaciona aún más los distintos modelos de árbol creados.

In [77]:
from sklearn.ensemble import RandomForestRegressor

In [78]:
from sklearn.metrics import mean_absolute_error

In [79]:
# Inicializo el modelo
regresor = RandomForestRegressor(criterion='absolute_error', random_state=42)

In [80]:
# Variables predictoras
x = students.drop(
    ['math score', 'reading score', 'writing score', 'puntaje_final'],
    axis=1
)

# Convertir variables categóricas a columnas numéricas
x = pd.get_dummies(x, dtype=int)

# Variable objetivo
y = students['puntaje_final']

# Separar los datos
X_train, X_test, y_train, y_test = train_test_split(
    x, y, test_size=0.2, random_state=42
)

# Entrenar
regresor.fit(X_train, y_train)

,"criterion criterion: {""squared_error"", ""absolute_error"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""absolute_error"" for the meanabsolute error, which minimizes the L1 loss using the median of each terminalnode, and ""poisson"" which uses reduction in Poisson deviance to find splits,also using the mean of each terminal node... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion... versionchanged:: 1.9 Criterion `""friedman_mse""` was deprecated.",'absolute_error'
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease 

In [81]:
# Entreno el modelo
regresor.fit(X_train, y_train);

In [82]:
regresor.get_params()

{'bootstrap': True,
 'ccp_alpha': 0.0,
 'criterion': 'absolute_error',
 'max_depth': None,
 'max_features': 1.0,
 'max_leaf_nodes': None,
 'max_samples': None,
 'min_impurity_decrease': 0.0,
 'min_samples_leaf': 1,
 'min_samples_split': 2,
 'min_weight_fraction_leaf': 0.0,
 'monotonic_cst': None,
 'n_estimators': 100,
 'n_jobs': None,
 'oob_score': False,
 'random_state': 42,
 'verbose': 0,
 'warm_start': False}

Ahora evaluaremos el modelo (Profundizaremos este concepto más adelante). Primero, hacemos predicciones para el set de evaluación y luego lo comparamos con los valores reales.

In [83]:
# Predigo los valores para el set de testeo
y_pred = regresor.predict(X_test)

In [84]:
# Calculo el error medio absoluto
mean_absolute_error(y_test, y_pred)

7.8857083333333335

### Support Vector Machine

Recordemos que el SVM es un algoritmo que establece un hiperplano que separa los puntos maximizando el margen. A su vez, una de sus ventajas es la aplicación de Kernels que me permite ajustar mejor el modelo.


In [85]:
from sklearn.svm import SVR

In [86]:
# Inicializo el modelo
regresor_svr = SVR(C=1.0, epsilon=0.2)

In [87]:
# Entreno el modelo
regresor_svr.fit(X_train, y_train);

In [88]:
regresor_svr.get_params()

{'C': 1.0,
 'cache_size': 200,
 'coef0': 0.0,
 'degree': 3,
 'epsilon': 0.2,
 'gamma': 'scale',
 'kernel': 'rbf',
 'max_iter': -1,
 'shrinking': True,
 'tol': 0.001,
 'verbose': False}

Ahora evaluaremos el modelo (Profundizaremos este concepto más adelante). Primero, hacemos predicciones para el set de evaluación y luego lo comparamos con los valores reales.

In [89]:
# Predigo los valores para el set de testeo
y_pred_svr = regresor_svr.predict(X_test)

In [90]:
# Calculo el error medio absoluto
mean_absolute_error(y_test, y_pred_svr)

8.30685462875379